# PesaGuard: Model Training, Evaluation & Calibration

This notebook handles Phase 2 of **PesaGuard**. We train:
1. **Isolation Forest** (unsupervised anomaly detector)
2. **XGBoost Classifier** (supervised model, optimized for class imbalance)
3. **Ensemble Scoring & Calibration** (weighted combination of both models)

We evaluate using Precision-Recall curves, compute SHAP explainability, and calibrate probabilities.

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix, classification_report
from sklearn.calibration import CalibratedClassifierCV
import shap

print('Imports complete!')

## 1. Load Processed Data

We load the engineered features saved in the previous notebook.

In [ ]:
X_train = pd.read_csv('data/processed/X_train_feats.csv')
X_test = pd.read_csv('data/processed/X_test_feats.csv')
y_train = pd.read_csv('data/processed/y_train.csv').iloc[:, 0]
y_test = pd.read_csv('data/processed/y_test.csv').iloc[:, 0]

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

## 2. Train Isolation Forest Anomaly Detector

Isolation Forest identifies outliers. It outputs an anomaly score (higher score = more anomalous).

In [ ]:
# Isolation Forest only requires feature matrix (unsupervised)
iforest = IsolationForest(n_estimators=100, contamination=0.01, random_state=42, n_jobs=-1)
iforest.fit(X_train)

# Get anomaly scores (Isolation Forest outputs decision_function where more negative = more anomalous)
# We invert and scale it to [0, 1] range where 1 is highly anomalous
def scale_anomaly_scores(model, X):
    scores = model.decision_function(X)
    # Map scores so that higher value = more anomalous
    # Raw decision function outputs range roughly from -0.5 to +0.5
    scaled = 1.0 - (scores - scores.min()) / (scores.max() - scores.min() + 1e-5)
    return scaled

train_anomaly = scale_anomaly_scores(iforest, X_train)
test_anomaly = scale_anomaly_scores(iforest, X_test)

print(f'Isolation Forest trained! Sample test anomaly scores: {test_anomaly[:5]}')

## 3. Train XGBoost Supervised Classifier

We handle class imbalance using XGBoost's `scale_pos_weight` parameter which sets the weight of positive class (fraud) proportional to the negative class ratio.

In [ ]:
# Calculate class ratio for scale_pos_weight
neg_count = sum(y_train == 0)
pos_count = sum(y_train == 1)
scale_weight = neg_count / (pos_count + 1e-5)
print(f'Imbalance scale weight: {scale_weight:.2f}')

# Train XGBClassifier
xgb = XGBClassifier(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train)

test_probs = xgb.predict_proba(X_test)[:, 1]
print('XGBoost trained!')

## 4. Ensemble Scoring: PesaGuard Score

We combine supervised and unsupervised models:
$$\text{Ensemble Score} = 0.7 \times \text{XGB Prob} + 0.3 \times \text{Anomaly Score}$$
Let's write this function and evaluate the performance of XGBoost, Isolation Forest, and the Ensemble.

In [ ]:
def compute_ensemble_score(xgb_prob, anomaly_score):
    return 0.7 * xgb_prob + 0.3 * anomaly_score

ensemble_probs = compute_ensemble_score(test_probs, test_anomaly)

# Metrics comparisons
for name, y_scores in [('XGBoost Only', test_probs), ('Anomaly Only', test_anomaly), ('Ensemble', ensemble_probs)]:
    auc = roc_auc_score(y_test, y_scores)
    pr_auc = average_precision_score(y_test, y_scores)
    print(f'{name:<15} | ROC-AUC: {auc:.4f} | PR-AUC (Average Precision): {pr_auc:.4f}')

In [ ]:
# Plot Precision-Recall Trade-off Curve
precision, recall, thresholds = precision_recall_curve(y_test, ensemble_probs)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision, label=f'Ensemble (PR-AUC = {average_precision_score(y_test, ensemble_probs):.4f})', color='green')
plt.xlabel('Recall (Detection Rate)')
plt.ylabel('Precision (True Fraud Rate in Flags)')
plt.title('Precision-Recall Curve (Ensemble Model)')
plt.legend(loc='best')
plt.show()

## 5. Probability Calibration

XGBoost trained with `scale_pos_weight` shifts output probabilities upwards (exaggerating true fraud probability). We use Isotonic Regression (CalibratedClassifierCV) to calibrate the probabilities so the outputs represent actual risk percentages.

In [ ]:
# Calibrate XGBoost
calibrated_xgb = CalibratedClassifierCV(estimator=xgb, method='isotonic', cv='prefit')
calibrated_xgb.fit(X_test, y_test) # prefitted calibrator on hold-out set

cal_test_probs = calibrated_xgb.predict_proba(X_test)[:, 1]
cal_ensemble = compute_ensemble_score(cal_test_probs, test_anomaly)

print(f'Original mean prediction prob: {test_probs.mean():.4f}')
print(f'Calibrated mean prediction prob: {cal_test_probs.mean():.4f}')
print(f'Actual fraud rate in test set: {y_test.mean():.4f}')

## 6. Model Explainability using SHAP

Explain predictions to analysts via SHAP waterfall charts. We will construct a TreeExplainer for XGBoost.

In [ ]:
# Shap analysis
explainer = shap.TreeExplainer(xgb)
# We take a sample of test rows to speed up explanation
sample_X = X_test.head(100)
shap_values = explainer(sample_X)

# Summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_X, show=False)
plt.title('SHAP Feature Importance Summary for XGBoost')
plt.tight_layout()
plt.show()

## 7. Save Model Weights & Pipeline Artifacts

We save our trained model objects to the `models/` directory for deployment in our FastAPI microservice.

In [ ]:
if not os.path.exists('models'):
    os.makedirs('models')

joblib.dump(iforest, 'models/iforest_model.joblib')
joblib.dump(calibrated_xgb, 'models/xgb_model.joblib')
# Note: We will export the feature pipeline object we learn in the previous notebook as well
print('Trained models saved successfully!')